In [ ]:
# Install required libraries
!pip -q install pandas numpy scikit-learn transformers datasets accelerate torch seaborn matplotlib tqdm

In [ ]:
# Imports and reproducibility
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, roc_auc_score
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set(style='whitegrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [ ]:
# Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/multisocial_outputs'
TEST_PATH = os.path.join(BASE_DIR, 'multisocial_test.csv')
ROC_PLOT_PATH = os.path.join(BASE_DIR, 'statistical/roc_combined.png')
RESULTS_JSON = os.path.join(BASE_DIR, 'statistical/results_statistical_per_language.json')

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
MAX_LENGTH = 512
BATCH_SIZE = 8

assert os.path.exists(TEST_PATH), f'Test file not found: {TEST_PATH}'

# CSV output directories
RESULTS_DIR = '/content/results'
DRIVE_RESULTS_DIR = os.path.join(BASE_DIR, 'statistical')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f'Test path: {TEST_PATH}')

Mounted at /content/drive
Test path: /content/drive/MyDrive/multisocial_outputs/multisocial_test.csv


In [ ]:
# Load and validate test data
test_df = pd.read_csv(TEST_PATH)
required_cols = {'text', 'label', 'language'}
missing = required_cols - set(test_df.columns)
if missing:
    raise ValueError(f'Missing required columns in test CSV: {sorted(missing)}')

test_df = test_df.dropna(subset=['text', 'label', 'language']).copy()
test_df['text'] = test_df['text'].astype(str)
test_df['label'] = test_df['label'].astype(int)
test_df['language'] = test_df['language'].astype(str)

assert not test_df.empty, 'Test dataframe is empty after cleanup.'

print(f'Test rows: {len(test_df)}')
print('Rows per language:')
print(test_df['language'].value_counts())

for lang in ['en', 'vi', 'zh', 'ar']:
    count = int((test_df['language'] == lang).sum())
    print(f'Language {lang}: {count} rows')
    assert count > 0, f'No test rows for language: {lang}'

Test rows: 3197
Rows per language:
language
en    800
zh    800
ar    800
vi    797
Name: count, dtype: int64
Language en: 800 rows
Language vi: 797 rows
Language zh: 800 rows
Language ar: 800 rows


In [ ]:
# Load tokenizer and model (bfloat16 + auto device mapping)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Model loaded and set to eval mode.')

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded and set to eval mode.


In [ ]:
# Perplexity computation functions
def compute_perplexity(texts, model, tokenizer, max_length=512, batch_size=8):
    perplexities = []

    for i in tqdm(range(0, len(texts), batch_size), desc='Computing perplexity'):
        batch_texts = texts[i:i + batch_size]

        try:
            enc = tokenizer(
                batch_texts,
                return_tensors='pt',
                truncation=True,
                padding=True,
                max_length=max_length,
            )

            enc = {k: v.to(model.device) for k, v in enc.items()}
            input_ids = enc['input_ids']
            attention_mask = enc['attention_mask']

            with torch.no_grad():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
                logits = outputs.logits

                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = input_ids[:, 1:].contiguous()
                shift_mask = attention_mask[:, 1:].contiguous()

                loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
                token_losses = loss_fct(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1),
                )
                token_losses = token_losses.view(shift_labels.size())

                seq_loss = (token_losses * shift_mask).sum(dim=1) / shift_mask.sum(dim=1).clamp(min=1)
                seq_ppl = torch.exp(seq_loss)

            perplexities.extend(seq_ppl.float().detach().cpu().numpy().tolist()) # Cast to float before numpy()

        except torch.cuda.OutOfMemoryError:
            print('OOM in batch. Falling back to per-sample computation for this batch...')
            torch.cuda.empty_cache()
            for text in batch_texts:
                enc1 = tokenizer(
                    [text],
                    return_tensors='pt',
                    truncation=True,
                    padding=True,
                    max_length=max_length,
                )
                enc1 = {k: v.to(model.device) for k, v in enc1.items()}

                with torch.no_grad():
                    out1 = model(**enc1, labels=enc1['input_ids'])
                    ppl1 = torch.exp(out1.loss)
                perplexities.append(float(ppl1.float().detach().cpu().item())) # Cast to float before item()

    ppls = np.array(perplexities, dtype=np.float32)
    assert len(ppls) == len(texts), 'Perplexity length mismatch.'
    assert np.isfinite(ppls).all(), 'Found non-finite perplexity values.'
    return ppls

In [ ]:
# Compute perplexities for all test texts
all_texts = test_df['text'].tolist()
all_ppl = compute_perplexity(
    texts=all_texts,
    model=model,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
)

test_eval_df = test_df.copy().reset_index(drop=True)
test_eval_df['perplexity'] = all_ppl
test_eval_df['machine_score'] = -test_eval_df['perplexity']

assert test_eval_df['perplexity'].notna().all(), 'Missing perplexity values found.'
print('Perplexity computation complete.')

Computing perplexity:   0%|          | 0/400 [00:00<?, ?it/s]

Perplexity computation complete.


In [ ]:
# Per-language ROC/AUC + sanity statistics
warnings.filterwarnings('ignore', category=UserWarning)

def safe_roc_auc(y_true, y_score):
    """Compute ROC-AUC; return NaN if single-class."""
    if len(np.unique(y_true)) < 2:
        print('  Warning: single class present — AUC undefined, returning NaN')
        return float('nan')
    return float(roc_auc_score(y_true, y_score))

def save_df(df, basename):
    """Save DataFrame as CSV to both local and Drive directories."""
    local_path = os.path.join(RESULTS_DIR, basename)
    drive_path = os.path.join(DRIVE_RESULTS_DIR, basename)
    df.to_csv(local_path, index=False)
    df.to_csv(drive_path, index=False)
    print(f'  Saved: {local_path}')
    print(f'  Saved: {drive_path}')

auc_results = []
roc_data = {}

for lang in ['en', 'vi', 'zh', 'ar']:
    lang_df = test_eval_df[test_eval_df['language'] == lang].copy()
    assert not lang_df.empty, f'No rows available for language {lang}'

    y_true = lang_df['label'].values.astype(int)
    y_score = lang_df['machine_score'].values.astype(np.float32)

    # ROC metrics
    auc = safe_roc_auc(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_data[lang] = (fpr, tpr)

    # Sanity-check averages by class
    human_mean_ppl = float(lang_df[lang_df['label'] == 0]['perplexity'].mean())
    machine_mean_ppl = float(lang_df[lang_df['label'] == 1]['perplexity'].mean())
    ppl_ratio = machine_mean_ppl / human_mean_ppl if human_mean_ppl > 0 else float('nan')

    auc_results.append({
        'language': lang,
        'auc_roc': float(auc),
        'n_samples': int(len(lang_df)),
        'mean_ppl_human': human_mean_ppl,
        'mean_ppl_machine': machine_mean_ppl,
        'ppl_ratio': float(ppl_ratio),
    })

    print(f'[{lang}] AUC: {auc:.4f} | mean PPL human: {human_mean_ppl:.3f} | mean PPL machine: {machine_mean_ppl:.3f} | ratio: {ppl_ratio:.2f}')

# Overall AUC
y_true_all = test_eval_df['label'].values.astype(int)
y_score_all = test_eval_df['machine_score'].values.astype(np.float32)
overall_auc = safe_roc_auc(y_true_all, y_score_all)
human_mean_all = float(test_eval_df[test_eval_df['label'] == 0]['perplexity'].mean())
machine_mean_all = float(test_eval_df[test_eval_df['label'] == 1]['perplexity'].mean())
auc_results.append({
    'language': 'overall',
    'auc_roc': float(overall_auc),
    'n_samples': int(len(test_eval_df)),
    'mean_ppl_human': human_mean_all,
    'mean_ppl_machine': machine_mean_all,
    'ppl_ratio': float(machine_mean_all / human_mean_all if human_mean_all > 0 else float('nan')),
})
print(f'[overall] AUC: {overall_auc:.4f}')

auc_df = pd.DataFrame(auc_results)
display(auc_df)

assert len(auc_df) == 5, 'Expected AUC results for 4 languages + overall.'
assert np.isfinite(auc_df['auc_roc']).all(), 'Non-finite AUC values found.'

[en] AUC: 0.7537 | mean PPL human: 1193.555 | mean PPL machine: 187.095 | ratio: 0.16
[vi] AUC: 0.5378 | mean PPL human: 1210.433 | mean PPL machine: 796.025 | ratio: 0.66
[zh] AUC: 0.5341 | mean PPL human: 221.514 | mean PPL machine: 207.885 | ratio: 0.94
[ar] AUC: 0.7784 | mean PPL human: 260.731 | mean PPL machine: 76.653 | ratio: 0.29
[overall] AUC: 0.6527


,language,auc_roc,n_samples,mean_ppl_human,mean_ppl_machine,ppl_ratio
0,en,0.753672,800,1193.554810,187.095306,0.156755
1,vi,0.537824,797,1210.432861,796.025269,0.657637
2,zh,0.534106,800,221.513748,207.884842,0.938474
3,ar,0.778447,800,260.730743,76.653168,0.293994
4,overall,0.652742,3197,726.398376,311.156830,0.428356


In [ ]:
# Save statistical results as CSV
print('Saving statistical results as CSV...')

# 1) Per-language AUC-ROC + PPL stats
save_df(auc_df, 'statistical_per_language.csv')

# 2) Test eval with labels, perplexity, machine_score
eval_cols = ['text', 'label', 'language', 'perplexity', 'machine_score']
save_df(test_eval_df[eval_cols], 'statistical_test_eval.csv')

# 3) Overall summary row
overall_row = auc_df[auc_df['language'] == 'overall'].copy()
save_df(overall_row, 'statistical_overall.csv')

print('CSV export complete.')

Saving statistical results as CSV...
  Saved: /content/results/statistical_per_language.csv
  Saved: /content/drive/MyDrive/multisocial_outputs/statistical/statistical_per_language.csv
  Saved: /content/results/statistical_test_eval.csv
  Saved: /content/drive/MyDrive/multisocial_outputs/statistical/statistical_test_eval.csv
  Saved: /content/results/statistical_overall.csv
  Saved: /content/drive/MyDrive/multisocial_outputs/statistical/statistical_overall.csv
CSV export complete.


In [ ]:
# Plot combined ROC curve and save
plt.figure(figsize=(7, 6))
for lang in ['en', 'vi', 'zh', 'ar']:
    fpr, tpr = roc_data[lang]
    auc_val = auc_df.loc[auc_df['language'] == lang, 'auc_roc'].iloc[0]
    plt.plot(fpr, tpr, label=f'{lang} (AUC={auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.7)
plt.title('Combined ROC Curves by Language (Score = -Perplexity)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(ROC_PLOT_PATH, dpi=220)
plt.close()

assert os.path.exists(ROC_PLOT_PATH), f'ROC plot was not saved: {ROC_PLOT_PATH}'
print(f'Saved combined ROC plot to: {ROC_PLOT_PATH}')

Saved combined ROC plot to: /content/drive/MyDrive/multisocial_outputs/statistical/roc_combined.png


In [ ]:
# Save results and cleanup
payload = {
    'seed': SEED,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'results': auc_df.to_dict(orient='records'),
    'roc_plot_path': ROC_PLOT_PATH,
}

with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

assert os.path.exists(RESULTS_JSON), f'Results JSON was not saved: {RESULTS_JSON}'
print(f'Saved statistical results to: {RESULTS_JSON}')

# Cleanup
del model
del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Cleanup complete. Done.')

Saved statistical results to: /content/drive/MyDrive/multisocial_outputs/statistical/results_statistical_per_language.json
Cleanup complete. Done.
